# Model Training — UNSW-NB15 Intrusion Detection

This notebook trains and cross-validates two classifiers — a Decision Tree and a Random Forest — to predict whether a network connection is `Normal` (0) or an `Attack` (1), using the cleaned, encoded, and scaled features produced in `02_preprocessing.ipynb`.

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

DATA_DIR = Path.cwd().parent / "data"
MODELS_DIR = Path.cwd().parent / "models"
MODELS_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train_clean.csv")
test_df = pd.read_csv(DATA_DIR / "test_clean.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (82332, 62)
Test shape: (175341, 62)


## Preparing features and target

The target is the binary `label` column (0 = normal, 1 = attack). `attack_cat_encoded` is dropped from the feature set — as explained in the preprocessing notebook, it is derived from the same ground truth as `label`, so including it would leak the answer directly into the model.

In [2]:
drop_cols = ["label", "attack_cat_encoded"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["label"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["label"]

print("X_train:", X_train.shape, " y_train balance:\n", y_train.value_counts(normalize=True))

X_train: (82332, 60)  y_train balance:
 label
1    0.5506
0    0.4494
Name: proportion, dtype: float64


## Model 1 — Decision Tree

**Why it's a reasonable choice for intrusion classification:**

- **Interpretability:** A single decision tree can be visualized and read top-to-bottom as a set of human-readable rules (e.g. "if `sttl` < X and `service` = http then normal"). For a security team, being able to explain *why* a connection was flagged is often as important as the flag itself — it supports triage and builds trust in the tool.
- **Handles mixed feature types natively:** Our feature set mixes continuous traffic metrics (byte counts, durations) with encoded categorical fields (protocol, service, state). Trees split on thresholds regardless of whether the underlying meaning is continuous or categorical, so they don't require features to be on the same scale or distribution to work correctly.
- **Fast baseline:** It trains quickly and gives us an immediately interpretable baseline to compare a more complex ensemble model against.
- **Known weakness:** A single tree is prone to overfitting the training data (memorizing noise), which is exactly what the Random Forest below is designed to address.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dt_model = DecisionTreeClassifier(random_state=42)

dt_cv_results = cross_validate(
    dt_model, X_train, y_train, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
)

print("Decision Tree — 5-fold cross-validation (training set):")
for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = dt_cv_results[f"test_{metric}"]
    print(f"  {metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Decision Tree — 5-fold cross-validation (training set):
  accuracy: 0.9669 (+/- 0.0014)
  precision: 0.9694 (+/- 0.0019)
  recall: 0.9705 (+/- 0.0018)
  f1: 0.9699 (+/- 0.0013)


In [4]:
# Fit the final Decision Tree on the full training set.
dt_model.fit(X_train, y_train)

dt_test_score = dt_model.score(X_test, y_test)
print("Decision Tree — held-out test set accuracy:", round(dt_test_score, 4))

Decision Tree — held-out test set accuracy: 0.895


## Model 2 — Random Forest

**Why it's a reasonable choice for intrusion classification:**

- **Robust to noisy features:** Network traffic data is full of features that are only weakly predictive or noisy for certain attack types. By averaging many trees, each trained on a random subset of data and features (bagging), a Random Forest cancels out the noise that would otherwise mislead a single tree, and is far less prone to overfitting.
- **Handles mixed feature types natively:** Like the Decision Tree, it works directly on a mix of scaled numeric and encoded categorical features without extra transformation.
- **Better generalization:** In practice, Random Forests almost always outperform a single Decision Tree on held-out data because the ensemble smooths over the idiosyncrasies any one tree would memorize — important for catching attack patterns not seen during training.
- **Feature importance:** It provides a ranked view of which traffic features drive its decisions, which is useful for a security analyst wanting to understand what signals matter most (e.g. is TTL, byte volume, or protocol the biggest tell?).
- **Trade-off:** It is less directly interpretable than a single tree (it's an ensemble of hundreds of trees), and slower to train and to run at inference time — a cost worth paying here for the accuracy gain.

In [5]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

rf_cv_results = cross_validate(
    rf_model, X_train, y_train, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
)

print("Random Forest — 5-fold cross-validation (training set):")
for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = rf_cv_results[f"test_{metric}"]
    print(f"  {metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Random Forest — 5-fold cross-validation (training set):
  accuracy: 0.9784 (+/- 0.0006)
  precision: 0.9844 (+/- 0.0016)
  recall: 0.9762 (+/- 0.0019)
  f1: 0.9803 (+/- 0.0005)


In [6]:
# Fit the final Random Forest on the full training set.
rf_model.fit(X_train, y_train)

rf_test_score = rf_model.score(X_test, y_test)
print("Random Forest — held-out test set accuracy:", round(rf_test_score, 4))

Random Forest — held-out test set accuracy: 0.9013


## Comparing the two models

The table below summarizes cross-validated training performance alongside held-out test accuracy, so we can see both how consistent each model is across folds and how well it generalizes to unseen traffic.

In [7]:
summary = pd.DataFrame({
    "Decision Tree": {
        "CV Accuracy": dt_cv_results["test_accuracy"].mean(),
        "CV Precision": dt_cv_results["test_precision"].mean(),
        "CV Recall": dt_cv_results["test_recall"].mean(),
        "CV F1": dt_cv_results["test_f1"].mean(),
        "Test Accuracy": dt_test_score,
    },
    "Random Forest": {
        "CV Accuracy": rf_cv_results["test_accuracy"].mean(),
        "CV Precision": rf_cv_results["test_precision"].mean(),
        "CV Recall": rf_cv_results["test_recall"].mean(),
        "CV F1": rf_cv_results["test_f1"].mean(),
        "Test Accuracy": rf_test_score,
    },
}).T

summary.round(4)

,CV Accuracy,CV Precision,CV Recall,CV F1,Test Accuracy
Decision Tree,0.9669,0.9694,0.9705,0.9699,0.8950
Random Forest,0.9784,0.9844,0.9762,0.9803,0.9013


## Saving the trained models

Both fitted models are persisted with `joblib` into the `models/` folder, so they can be reloaded later for evaluation or deployment without retraining.

In [8]:
dt_path = MODELS_DIR / "decision_tree.joblib"
rf_path = MODELS_DIR / "random_forest.joblib"

joblib.dump(dt_model, dt_path)
joblib.dump(rf_model, rf_path)

print("Saved:", dt_path)
print("Saved:", rf_path)

Saved: D:\IS\ids-ml-project\models\decision_tree.joblib
Saved: D:\IS\ids-ml-project\models\random_forest.joblib
